# Model Training — Wildfire Risk Forecaster

Train XGBoost and Random Forest classifiers for wildfire risk prediction.
Evaluate with AUC-ROC and F1-score using stratified cross-validation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import xgboost as xgb
import matplotlib.pyplot as plt

## Modeling Steps

1. Load feature matrix and define target variable
2. Train/test split with stratification on fire occurrence
3. Train Random Forest baseline
4. Train XGBoost with hyperparameter tuning
5. Compare models on AUC-ROC and F1-score

In [ ]:
# Load feature matrix
df = pd.read_parquet('../data/processed/feature_matrix.parquet')
feature_cols = [c for c in df.columns if c not in ['grid_id', 'date', 'fire_occurred']]
X = df[feature_cols].values
y = df['fire_occurred'].values
print(f'Features: {X.shape[1]}, Samples: {X.shape[0]}')
print(f'Class balance: {y.mean():.3f} positive rate')

In [ ]:
# Random Forest baseline
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_auc = cross_val_score(rf, X, y, cv=skf, scoring='roc_auc')
print(f'RF AUC-ROC: {rf_auc.mean():.4f} +/- {rf_auc.std():.4f}')

# XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_auc = cross_val_score(xgb_clf, X, y, cv=skf, scoring='roc_auc')
print(f'XGB AUC-ROC: {xgb_auc.mean():.4f} +/- {xgb_auc.std():.4f}')